# AI4I 2020 — Exploratory Data Analysis

This notebook explores a **synthetic**, non-time-series classification dataset. No predictive metrics are calculated here; no leakage columns are used for modelling.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd().resolve()
if not (root / "src").is_dir():
    root = root.parent
assert (root / "src").is_dir(), "Open this notebook from the project or notebooks folder"
sys.path.insert(0, str(root))
from src.data import load_data, inspect_quality
from src.config import TARGET, NUMERIC_FEATURES, FEATURES
df = load_data(root / "data" / "raw" / "ai4i2020.csv")
print(df.shape)
display(df.head())

## Data quality
Check missing values, duplicate identifiers, column types and class frequency before modelling.

In [ ]:
quality = inspect_quality(df)
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
print("Duplicate rows:", quality["duplicate_rows"])
print("Duplicate UDI:", quality["duplicate_udi"])
print("Target counts:", quality["target_counts"])

## Target distribution
A low positive class rate motivates recall, precision and average precision rather than accuracy alone.

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(6,4))
counts = df[TARGET].value_counts().reindex([0, 1], fill_value=0)
ax.bar(counts.index.astype(str), counts.values)
ax.set_title("Observed machine-failure labels")
plt.show()
print("Observed failure %:", round(df[TARGET].mean()*100, 2))

## Product type and labelled failure
Compare observed failure rates descriptively, without assuming that product type causes failures.

In [ ]:
display(df.groupby("Type")[TARGET].agg(["count", "sum", "mean"]))
fig, ax = plt.subplots(figsize=(6,4))
(df.groupby("Type")[TARGET].mean()*100).plot.bar(ax=ax)
ax.set(ylabel="Labelled failures (%)", title="Observed failure rate by product type")
plt.show()

## Sensor profiles
Compare the distributions for positive and negative labels. Patterns in synthetic data do not imply causal maintenance mechanisms.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, column in zip(axes.flat, NUMERIC_FEATURES):
    sns.histplot(data=df, x=column, hue=TARGET, bins=30, stat="density", common_norm=False, element="step", ax=ax)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## Correlations
Pearson correlation summarises linear relationships only. Failure-mode flags are deliberately not included in the heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
sns.heatmap(df[NUMERIC_FEATURES + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.tight_layout()
plt.show()